In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 讀取資料
df = pd.read_csv(r'C:\Users\ian32\Downloads\final\data\train.csv', nrows=1000)

def parse_ad_slot_id_v3(slot_id):
    """
    一個更穩健的 ad_slot_id 解析函式 v3
    """
    # 預設所有欄位為 None
    result = {
        'slot_p1': None, 'slot_p2': None, 'slot_p3': None,
        'slot_platform': None, 'slot_platform_id1': None, 'slot_platform_id2': None,
        'slot_category': None, 'slot_flag': None, 'slot_width_tier': None,
        'slot_numeric_id': None, # 新增：用於純數字ID
        'slot_unknown_id': None
    }
    
    s = str(slot_id)

    # 規則A: mm_ 開頭 (非常具體，優先處理)
    if s.startswith('mm_'):
        parts = s.split('_')
        if len(parts) == 4:
            result['slot_p1'] = parts[1]
            result['slot_p2'] = parts[2]
            result['slot_p3'] = parts[3]
        else: # 處理格式不符的 mm_ ID
            result['slot_unknown_id'] = s
            
    # 規則B: 已知平台開頭 (也很具體)
    elif any(s.startswith(prefix) for prefix in ['discuz_', 'phpwind_', 'dz_', 'pw_']):
        parts = s.split('_')
        result['slot_platform'] = parts[0]
        if len(parts) > 1: result['slot_platform_id1'] = parts[1]
        if len(parts) > 2: result['slot_platform_id2'] = parts[2]

    # 規則C: 純數字ID (非常重要的模式)
    elif s.isdigit():
        result['slot_numeric_id'] = s

    # 規則D: 描述性格式 (較為模糊，放在後面)
    elif '_' in s: # 作為一個較通用的規則
        parts = s.split('_')
        result['slot_category'] = parts[0]
        # 遍歷剩餘部分尋找 flag 和 width tier
        remaining_parts = parts[1:]
        temp_flag = []
        for part in remaining_parts:
            if part.startswith('Width'):
                result['slot_width_tier'] = part
            # 假設單一大寫字母為 flag
            elif len(part) == 1 and part.isupper():
                temp_flag.append(part)
        if temp_flag:
            result['slot_flag'] = '_'.join(temp_flag)

    # 後備規則: 處理剩餘所有無法識別的格式
    else:
        result['slot_unknown_id'] = s
        
    return pd.Series(result)

# 解析 ad_slot_id
print("--- 開始解析 ad_slot_id ---")
df_adslot = df['ad_slot_id'].apply(parse_ad_slot_id_v3)
df = pd.concat([df.reset_index(drop=True), df_adslot], axis=1)
print("--- 解析完成 ---")

# 對所有新特徵進行標籤編碼
new_feature_columns = [
    'slot_p1', 'slot_p2', 'slot_p3', 
    'slot_platform', 'slot_platform_id1', 'slot_platform_id2',
    'slot_category', 'slot_flag', 'slot_width_tier', 
    'slot_numeric_id', 'slot_unknown_id'
]

print("--- 開始進行標籤編碼 ---")
for col in new_feature_columns:
    if col in df.columns:
        le = LabelEncoder()
        df[col + '_label'] = le.fit_transform(df[col].astype(str))
print("--- 標籤編碼完成 ---")

# 顯示處理後的結果
display_columns = ['ad_slot_id'] + new_feature_columns + [col + '_label' for col in ['slot_p1', 'slot_numeric_id', 'slot_category']]
print("\n--- 最終處理結果預覽 ---")
df[display_columns]

--- 開始解析 ad_slot_id ---
--- 解析完成 ---
--- 開始進行標籤編碼 ---
--- 標籤編碼完成 ---

--- 最終處理結果預覽 ---


,ad_slot_id,slot_p1,slot_p2,slot_p3,slot_platform,slot_platform_id1,slot_platform_id2,slot_category,slot_flag,slot_width_tier,slot_numeric_id,slot_unknown_id,slot_p1_label,slot_numeric_id_label,slot_category_label
0,mm_34022157_3445226_11175096,34022157,3445226,11175096,None,None,None,None,None,None,None,None,71,0,8
1,mm_13991432_2298120_9467354,13991432,2298120,9467354,None,None,None,None,None,None,None,None,42,0,8
2,mm_10024662_3445902_11178359,10024662,3445902,11178359,None,None,None,None,None,None,None,None,9,0,8
3,mm_10024662_3445902_11178345,10024662,3445902,11178345,None,None,None,None,None,None,None,None,9,0,8
4,mm_12987374_1803472_13162557,12987374,1803472,13162557,None,None,None,None,None,None,None,None,36,0,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,News_Pic_Width1,None,None,None,None,None,None,News,None,Width1,None,None,81,0,7
996,mm_10027070_118039_9659846,10027070,118039,9659846,None,None,None,None,None,None,None,None,11,0,8
997,mm_10024662_3445902_11178345,10024662,3445902,11178345,None,None,None,None,None,None,None,None,9,0,8
998,mm_33805665_3431864_11230552,33805665,3431864,11230552,None,None,None,None,None,None,None,None,70,0,8
